# Finland Electricity Price Analysis — Data Cleaning

This notebook combines and cleans three raw datasets (ENTSO-E day-ahead prices, 
ENTSO-E generation by production type, Fingrid consumption) covering 2018–2026, 
preparing them for the main analysis: does Finland's generation mix (nuclear, wind) 
explain price movements around the 2022 energy crisis, or were external factors 
the bigger driver?

**This notebook covers:** combining yearly files, checking for duplicates/gaps, 
resolving timezone and resolution mismatches, and producing clean combined CSVs 
for each dataset.

In [1]:
import pandas as pd
import glob

In [5]:
# Step 1: find every file matching the pattern "price_*.csv" in your data folder
file_list = glob.glob('../data/raw/Actual_Generation_per_Production_Type/AGGREGATED_*.CSV')
print(file_list)

['../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201712312300-201812312300 (1).csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201812312300-201912312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201912312300-202012312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202012312300-202112312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202112312300-202212312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202212312300-202312312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202312312300-202412312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202412312300-202512

In [10]:
# Step 2: loop through each filename, read it into a table, and collect them in a list
all_years = []
for file in file_list:
    df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited
    all_years.append(df_year)

C:\Users\abhis\AppData\Local\Temp\ipykernel_756\323770808.py:4: DtypeWarning: Columns (0: Generation (MW)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited


In [20]:
# Step 3: stack all 8 tables into one combined table
df_price_combined = pd.concat(all_years, ignore_index=True)

In [21]:
# Step 4: check for duplicate timestamps and product type before trusting the result

duplicates = df_price_combined[df_price_combined.duplicated(subset=["MTU (CET/CEST)", "Production Type"], keep=False)]
print("Number of duplicate rows:", len(duplicates))

Number of duplicate rows: 0


In [24]:
df_price_combined.head()

,timestamp_raw,area,production_type,generation_mw
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Biomass,873.31
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Biomass,855.71
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Biomass,858.29
3,01/01/2018 03:00:00 - 01/01/2018 04:00:00,BZN|FI,Biomass,874.48
4,01/01/2018 04:00:00 - 01/01/2018 05:00:00,BZN|FI,Biomass,869.99


In [23]:
df_price_combined.columns = ['timestamp_raw', 'area', 'production_type','generation_mw']

In [26]:
df_price_combined.isnull().sum()

timestamp_raw       0
area                0
production_type     0
generation_mw      40
dtype: int64

In [29]:
print(df_price_combined[df_price_combined['generation_mw'].isnull()]['production_type'].value_counts())

production_type
Solar                             12
Fossil Gas                         3
Fossil Hard coal                   3
Fossil Oil                         3
Hydro Run-of-river and pondage     3
Other renewable                    3
Waste                              3
Biomass                            2
Fossil Peat                        2
Nuclear                            2
Other                              2
Wind Onshore                       2
Name: count, dtype: int64


In [31]:
# Sort first so forward fill works in the right direction
df_price_combined = df_price_combined.sort_values(['production_type', 'timestamp_raw'])

In [32]:
# Forward fill nulls within each production type separately
df_price_combined['generation_mw'] =df_price_combined.groupby('production_type')['generation_mw'].ffill()

In [33]:
# Confirm no nulls remain
df_price_combined['generation_mw'].isnull().sum()

np.int64(0)

In [36]:
df_price_combined['timestamp'] = df_price_combined['timestamp_raw'].str.split("-").str[0]

In [43]:
df_price_combined['timestamp']

0          01/01/2018 00:00:00
1          01/01/2018 01:00:00
2          01/01/2018 02:00:00
3          01/01/2018 03:00:00
4          01/01/2018 04:00:00
                  ...         
3313795    31/12/2026 22:45:00
3313796    31/12/2026 23:00:00
3313797    31/12/2026 23:15:00
3313798    31/12/2026 23:30:00
3313799    31/12/2026 23:45:00
Name: timestamp, Length: 3313800, dtype: object

In [ ]:
# Strip any timezone suffix like (CET), (CEST), etc.
df_price_combined['timestamp'] = (
    df_price_combined['timestamp']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)  # removes anything in parentheses
    .str.strip()
)

In [ ]:
# Convert to datetime
df_price_combined['timestamp'] = pd.to_datetime(
    df_price_combined['timestamp'],
    format='%d/%m/%Y %H:%M:%S',
    dayfirst=True
)

In [48]:
df_price_combined =df_price_combined.drop(columns='timestamp_raw')

In [ ]:
# Replace n/e and - with 0, then convert to float
df_price_combined['generation_mw'] = df_price_combined['generation_mw'].replace({'n/e':0, '-':0})

In [54]:
# convert to float

df_price_combined['generation_mw'] = pd.to_numeric(df_price_combined['generation_mw'])

In [55]:
df_price_combined.shape

(3313800, 4)

In [56]:
df_price_combined.dtypes

area                          str
production_type               str
generation_mw             float64
timestamp          datetime64[us]
dtype: object

In [57]:
df_price_combined.isnull().sum()

area               0
production_type    0
generation_mw      0
timestamp          0
dtype: int64

In [58]:
df_price_combined.head()

,area,production_type,generation_mw,timestamp
0,BZN|FI,Biomass,873.31,2018-01-01 00:00:00
1,BZN|FI,Biomass,855.71,2018-01-01 01:00:00
2,BZN|FI,Biomass,858.29,2018-01-01 02:00:00
3,BZN|FI,Biomass,874.48,2018-01-01 03:00:00
4,BZN|FI,Biomass,869.99,2018-01-01 04:00:00


In [59]:
print("Shape:", df_price_combined.shape)
print("\nData types:")
print(df_price_combined.dtypes)
print("\nNull values:")
print(df_price_combined.isnull().sum())
print("\nDate range:")
print("Start:", df_price_combined['timestamp'].min())
print("End:", df_price_combined['timestamp'].max())
print("\nProduction types:")
print(df_price_combined['production_type'].unique())

Shape: (3313800, 4)

Data types:
area                          str
production_type               str
generation_mw             float64
timestamp          datetime64[us]
dtype: object

Null values:
area               0
production_type    0
generation_mw      0
timestamp          0
dtype: int64

Date range:
Start: 2018-01-01 00:00:00
End: 2026-12-31 23:45:00

Production types:
<StringArray>
[                       'Biomass',                 'Energy storage',
      'Fossil Brown coal/Lignite',        'Fossil Coal-derived gas',
                     'Fossil Gas',               'Fossil Hard coal',
                     'Fossil Oil',               'Fossil Oil shale',
                    'Fossil Peat',                     'Geothermal',
           'Hydro Pumped Storage', 'Hydro Run-of-river and pondage',
          'Hydro Water Reservoir',                         'Marine',
                        'Nuclear',                          'Other',
                'Other renewable',                      

In [60]:
# Step 5: sort chronologically and save

price_combined = price_combined.sort_values(["MTU (CET/CEST)", "Production Type"])
price_combined.to_csv('../data/processed/finland_generation_mix_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(price_combined))

Saved successfully, total rows: 3313800
